In [1]:
# Core imports
import numpy as np
from pathlib import Path
import sys
import warnings
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)

# Base paths
PROJECT_ROOT = Path("..")
DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"

# Allow imports from ../src
sys.path.append(str(PROJECT_ROOT / "src"))

# Import our utilities
from bbo_utils import (
    load_xy,
    fit_gp,
    acquisition_ucb,
    generate_random_candidates,
    format_submission,
    validate_submission_string,
)

#from candidates import generate_random_candidates

In [2]:
for func_id in range(1, 9):
    X, y = load_xy(func_id, DATA_RAW_DIR)

    print(
        f"Function {func_id}: "
        f"X shape={X.shape}, "
        f"y shape={y.shape}, "
        f"X range=({X.min():.6f}, {X.max():.6f}), "
        f"y range=({y.min():.6f}, {y.max():.6f})"
    )

Function 1: X shape=(10, 2), y shape=(10,), X range=(0.078723, 0.883890), y range=(-0.003606, 0.000000)
Function 2: X shape=(10, 2), y shape=(10,), X range=(0.028698, 0.926564), y range=(-0.065624, 0.611205)
Function 3: X shape=(15, 3), y shape=(15,), X range=(0.046809, 0.990882), y range=(-0.398926, -0.034835)
Function 4: X shape=(30, 4), y shape=(30,), X range=(0.006250, 0.999483), y range=(-32.625660, -4.025542)
Function 5: X shape=(20, 4), y shape=(20,), X range=(0.038193, 0.957644), y range=(0.112940, 1088.859618)
Function 6: X shape=(20, 5), y shape=(20,), X range=(0.004911, 0.978806), y range=(-2.571170, -0.714265)
Function 7: X shape=(30, 6), y shape=(30,), X range=(0.003635, 0.998655), y range=(0.002701, 1.364968)
Function 8: X shape=(40, 8), y shape=(40,), X range=(0.003419, 0.998885), y range=(5.592193, 9.598482)


In [3]:
# Week 1 submitted inputs (portal)
WEEK1_INPUTS = {
    1: [0.705738, 0.871990],
    2: [0.828497, 0.970137],
    3: [0.418575, 0.366581, 0.521443],
    4: [0.370752, 0.426131, 0.337047, 0.465539],
    5: [0.398348, 0.896184, 0.926939, 0.972836],
    6: [0.296485, 0.322087, 0.332446, 0.760368, 0.040944],
    7: [0.033169, 0.329738, 0.366358, 0.234301, 0.286315, 0.704927],
    8: [0.147947, 0.193825, 0.060981, 0.008004, 0.719306, 0.453461, 0.045592, 0.938970],
}

# Week 1 observed outputs (email)
WEEK1_OUTPUTS = {
    1: 8.994904739436242e-46,
    2: 0.07492805470733854,
    3: -0.013221116845993724,
    4: -0.4713345507601363,
    5: 2360.0402820167205,
    6: -0.49886770748675036,
    7: 2.4608579737515917,
    8: 9.894432442301,
}

# Week 2 submitted inputs (portal)
WEEK2_INPUTS = {
    1: [0.763923, 0.799245],
    2: [0.692932, 0.920964],
    3: [0.388137, 0.056619, 0.408877],
    4: [0.413742, 0.470182, 0.290309, 0.439853],
    5: [0.901642, 0.984569, 0.982367, 0.952999],
    6: [0.347816, 0.270027, 0.588964, 0.987295, 0.290167],
    7: [0.196139, 0.369342, 0.400707, 0.230683, 0.233984, 0.607869],
    8: [0.078187, 0.158932, 0.026599, 0.138226, 0.529417, 0.076738, 0.059868, 0.118162],
}

# Week 2 observed outputs (email)
WEEK2_OUTPUTS = {
    1: 1.1451214454803375e-33,
    2: 0.5227732568346157,
    3: -0.08200213652671233,
    4: -1.7902558622959037,
    5: 5823.4794607079075,
    6: -0.6884064929199594,
    7: 2.425181219664537,
    8: 9.6885056366981,
}

# Week 3 submitted inputs (portal)
WEEK3_INPUTS = {
    1: [0.773956, 0.438878],
    2: [0.773956, 0.438878],
    3: [0.708862, 0.999390, 0.007483],
    4: [0.381611, 0.390023, 0.489693, 0.455284],
    5: [0.963141, 0.994441, 0.801624, 0.999286],
    6: [0.268756, 0.730815, 0.162103, 0.932093, 0.000434],
    7: [0.219388, 0.134546, 0.344610, 0.186044, 0.178171, 0.647537],
    8: [0.043177, 0.514701, 0.082182, 0.986416, 0.832375, 0.826257, 0.077916, 0.007706],
}

# Week 3 observed outputs (email)
WEEK3_OUTPUTS = {
    1: 9.656222973626152e-41,
    2: 0.19196354967373935,
    3: -0.1388977457468655,
    4: -2.0470834341233686,
    5: 5394.580680183886,
    6: -1.281015418338123,
    7: 2.23531139550533,
    8: 8.9822261407959,
}

WEEK4_INPUTS = {
    1: [0.763923, 0.799245],
    2: [0.692932, 0.920964],
    3: [0.388137, 0.056619, 0.408877],
    4: [0.413742, 0.470182, 0.290309, 0.439853],
    5: [0.901642, 0.984569, 0.982367, 0.952999],
    6: [0.347816, 0.270027, 0.588964, 0.987295, 0.290167],
    7: [0.196139, 0.369342, 0.400707, 0.230683, 0.233984, 0.607869],
    8: [0.078187, 0.158932, 0.026599, 0.138226, 0.529417, 0.076738, 0.059868, 0.118162],
}

WEEK4_OUTPUTS = {
    1: 1.539769610658716e-40,
    2: 0.44842891948127245,
    3: -0.10719535681363297,
    4: -34.17518209728457,
    5: 5823.4794607079075,
    6: -0.8811846371232062,
    7: 1.049154230940631,
    8: 9.1848478262995,
}

# Week 5 submitted inputs (portal)
WEEK5_INPUTS = {
    1: [0.652299, 0.043775],
    2: [0.718807, 0.999999],
    3: [0.451163, 0.676920, 0.592177],
    4: [0.905604, 0.077227, 0.272570, 0.621850],
    5: [0.980000, 0.980000, 0.980000, 0.980000],
    6: [0.208776, 0.001828, 0.479017, 0.537405, 0.000000],
    7: [0.099617, 0.221403, 0.313640, 0.206654, 0.271980, 0.685887],
    8: [0.062606, 0.794599, 0.071308, 0.396228, 0.403033, 0.973574, 0.035939, 0.983739],
}

# Week 5 observed outputs (email)
WEEK5_OUTPUTS = {
    1: 5.761308350753379e-137,
    2: 0.6099221798837782,
    3: -0.06776346304915429,
    4: -19.2635124768894,
    5: 7223.214754607814,
    6: -1.0312540160652932,
    7: 2.7151083404808167,
    8: 9.139111707676399,
}

# Week 6 submitted inputs (portal)
WEEK6_INPUTS = {
    1: [0.724738, 0.737753],
    2: [0.706859, 0.944063],
    3: [0.949400, 0.923150, 0.910354],
    4: [0.320914, 0.348196, 0.389118, 0.495202],
    5: [0.999999, 0.999999, 0.999999, 0.999999],
    6: [0.413869, 0.456014, 0.616938, 0.570196, 0.000000],
    7: [0.000000, 0.142926, 0.335344, 0.140577, 0.250679, 0.784978],
    8: [0.040611, 0.019643, 0.444982, 0.169826, 0.882469, 0.946076, 0.205009, 0.773192],
}

# Week 6 observed outputs (email)
WEEK6_OUTPUTS = {
    1: 9.80777872783949e-16,
    2: 0.6027157967768804,
    3: -0.11435005715863795,
    4: -1.5659945410762037,
    5: 8662.405001248297,
    6: -0.5136137545123074,
    7: 1.8800880725938567,
    8: 9.4724848018561,
}

In [4]:
def load_xy_with_weeks_1_to_6(func_id: int, data_raw_dir):
    """
    Build training dataset for Week 7:
    initial + Week 1 + Week 2 + Week 3 + Week 4 + Week 5 + Week 6
    """
    X0, y0 = load_xy(func_id, data_raw_dir)
    dim = X0.shape[1]

    # Week 1
    x_w1 = np.array(WEEK1_INPUTS[func_id], dtype=float).reshape(1, -1)
    y_w1 = np.array([WEEK1_OUTPUTS[func_id]], dtype=float)

    # Week 2
    x_w2 = np.array(WEEK2_INPUTS[func_id], dtype=float).reshape(1, -1)
    y_w2 = np.array([WEEK2_OUTPUTS[func_id]], dtype=float)

    # Week 3
    x_w3 = np.array(WEEK3_INPUTS[func_id], dtype=float).reshape(1, -1)
    y_w3 = np.array([WEEK3_OUTPUTS[func_id]], dtype=float)

    # Week 4
    x_w4 = np.array(WEEK4_INPUTS[func_id], dtype=float).reshape(1, -1)
    y_w4 = np.array([WEEK4_OUTPUTS[func_id]], dtype=float)

    # Week 5
    x_w5 = np.array(WEEK5_INPUTS[func_id], dtype=float).reshape(1, -1)
    y_w5 = np.array([WEEK5_OUTPUTS[func_id]], dtype=float)

    # Week 6
    x_w6 = np.array(WEEK6_INPUTS[func_id], dtype=float).reshape(1, -1)
    y_w6 = np.array([WEEK6_OUTPUTS[func_id]], dtype=float)

    # Dimension sanity check
    if any(x.shape[1] != dim for x in (x_w1, x_w2, x_w3, x_w4, x_w5, x_w6)):
        raise ValueError(f"Function {func_id}: dimension mismatch in weekly inputs.")

    # Combine
    X_updated = np.vstack([X0, x_w1, x_w2, x_w3, x_w4, x_w5, x_w6])
    y_updated = np.concatenate([y0, y_w1, y_w2, y_w3, y_w4, y_w5, y_w6])

    return X_updated, y_updated

In [5]:
def sanity_check_dataset(func_id: int, data_raw_dir):
    """
    Performs sanity checks on dataset built with Week 1–5 data.
    """

    print(f"\n===== SANITY CHECK: Function {func_id} =====")

    X, y = load_xy_with_weeks_1_to_6(func_id, data_raw_dir)

    # 1️⃣ Basic shape info
    print(f"Shape of X: {X.shape}")
    print(f"Shape of y: {y.shape}")

    # 2️⃣ Check for NaN or Inf
    if np.isnan(X).any() or np.isnan(y).any():
        print("❌ NaN detected!")
    else:
        print("✅ No NaN values detected.")

    if np.isinf(X).any() or np.isinf(y).any():
        print("❌ Infinite values detected!")
    else:
        print("✅ No infinite values detected.")

    # 3️⃣ Dimensional consistency
    dim = X.shape[1]
    print(f"Dimension (input size): {dim}")

    # 4️⃣ Best-so-far check
    best_idx = int(np.argmax(y))
    best_y = y[best_idx]
    best_x = X[best_idx]

    print(f"Best index so far: {best_idx}")
    print(f"Best y so far: {best_y}")
    print(f"Best x so far: {best_x}")

    # 5️⃣ Basic statistics
    print(f"y min: {np.min(y)}")
    print(f"y max: {np.max(y)}")
    print(f"y mean: {np.mean(y)}")
    print(f"Total points: {len(y)}")

    print("===== END SANITY CHECK =====\n")

In [6]:
for func_id in range(1, 9):
    sanity_check_dataset(func_id, DATA_RAW_DIR)


===== SANITY CHECK: Function 1 =====
Shape of X: (16, 2)
Shape of y: (16,)
✅ No NaN values detected.
✅ No infinite values detected.
Dimension (input size): 2
Best index so far: 15
Best y so far: 9.80777872783949e-16
Best x so far: [0.724738 0.737753]
y min: -0.0036060626443634764
y max: 9.80777872783949e-16
y mean: -0.00022537891527260777
Total points: 16
===== END SANITY CHECK =====


===== SANITY CHECK: Function 2 =====
Shape of X: (16, 2)
Shape of y: (16,)
✅ No NaN values detected.
✅ No infinite values detected.
Dimension (input size): 2
Best index so far: 9
Best y so far: 0.6112052157614438
Best x so far: [0.70263656 0.9265642 ]
y min: -0.06562362443733738
y max: 0.6112052157614438
y mean: 0.2973418387848155
Total points: 16
===== END SANITY CHECK =====


===== SANITY CHECK: Function 3 =====
Shape of X: (21, 3)
Shape of y: (21,)
✅ No NaN values detected.
✅ No infinite values detected.
Dimension (input size): 3
Best index so far: 15
Best y so far: -0.013221116845993724
Best x so fa

In [ ]:
def best_so_far_summary(func_id, data_raw_dir):
    X_all, y_all = load_xy_with_weeks_1_to_6(func_id, data_raw_dir)

    best_idx = int(np.argmax(y_all))
    best_val = float(y_all[best_idx])

    # Number of initial rows
    X0, y0 = load_xy(func_id, data_raw_dir)
    n0 = len(y0)

    if best_idx < n0:
        source = "Initial data"
    else:
        source = f"Week {best_idx - n0 + 1}"

    return {
        "Function": func_id,
        "Best_y": best_val,
        "Best_index": best_idx,
        "Source": source,
        "Total_points": len(y_all)
    }
    

import pandas as pd

rows = []
for func_id in range(1, 9):
    rows.append(best_so_far_summary(func_id, DATA_RAW_DIR))

df_best = pd.DataFrame(rows)
df_best    

,Function,Best_y,Best_index,Source,Total_points
0,1,7.710875e-16,2,Initial data,15
1,2,6.112052e-01,9,Initial data,15
2,3,-1.322112e-02,15,Week 1,20
3,4,-4.713346e-01,30,Week 1,35
4,5,7.223215e+03,24,Week 5,25
5,6,-4.988677e-01,20,Week 1,25
6,7,2.715108e+00,34,Week 5,35
7,8,9.894432e+00,40,Week 1,45


In [7]:
# =========================================
# Week 7 Strategy Settings
# =========================================
# mode meanings:
# - explore: high kappa, more global candidates
# - balanced: medium kappa, mix global/local
# - exploit: lower kappa, more local candidates
# - force: deterministic submission string

WEEK7_STRATEGY = {
    1: {"mode": "explore",  "kappa": 3.2, "n_candidates": 30000, "local_frac": 0.15, "local_sigma": 0.18},
    2: {"mode": "exploit",  "kappa": 1.2, "n_candidates": 18000, "local_frac": 0.75, "local_sigma": 0.05},
    3: {"mode": "balanced", "kappa": 2.0, "n_candidates": 22000, "local_frac": 0.40, "local_sigma": 0.10},
    4: {"mode": "exploit",  "kappa": 1.4, "n_candidates": 18000, "local_frac": 0.70, "local_sigma": 0.05},
    5: {"mode": "force",    "force_x": "0.999999-0.999999-0.999999-0.999999"},
    6: {"mode": "balanced", "kappa": 2.4, "n_candidates": 24000, "local_frac": 0.35, "local_sigma": 0.12},
    7: {"mode": "exploit",  "kappa": 1.2, "n_candidates": 18000, "local_frac": 0.70, "local_sigma": 0.05},
    8: {"mode": "balanced", "kappa": 2.0, "n_candidates": 22000, "local_frac": 0.50, "local_sigma": 0.08},
}

print("Week 7 strategy loaded.")
for fid in range(1, 9):
    print(f"Function {fid}: {WEEK7_STRATEGY[fid]}")

Week 7 strategy loaded.
Function 1: {'mode': 'explore', 'kappa': 3.2, 'n_candidates': 30000, 'local_frac': 0.15, 'local_sigma': 0.18}
Function 2: {'mode': 'exploit', 'kappa': 1.2, 'n_candidates': 18000, 'local_frac': 0.75, 'local_sigma': 0.05}
Function 3: {'mode': 'balanced', 'kappa': 2.0, 'n_candidates': 22000, 'local_frac': 0.4, 'local_sigma': 0.1}
Function 4: {'mode': 'exploit', 'kappa': 1.4, 'n_candidates': 18000, 'local_frac': 0.7, 'local_sigma': 0.05}
Function 5: {'mode': 'force', 'force_x': '0.999999-0.999999-0.999999-0.999999'}
Function 6: {'mode': 'balanced', 'kappa': 2.4, 'n_candidates': 24000, 'local_frac': 0.35, 'local_sigma': 0.12}
Function 7: {'mode': 'exploit', 'kappa': 1.2, 'n_candidates': 18000, 'local_frac': 0.7, 'local_sigma': 0.05}
Function 8: {'mode': 'balanced', 'kappa': 2.0, 'n_candidates': 22000, 'local_frac': 0.5, 'local_sigma': 0.08}


## Function 1 — Week 7 Suggestion

In [8]:
# ==========================================================
# Function 1 — Week 7 suggestion (explore wide)
# ==========================================================
func_id = 1
cfg = WEEK7_STRATEGY[func_id]

# 1) Load dataset up to Week 6
X_all, y_all = load_xy_with_weeks_1_to_6(func_id, DATA_RAW_DIR)
dim = X_all.shape[1]

print(f"Function {func_id} | points={len(y_all)} | dim={dim}")

# 2) Best-so-far
best_idx = int(np.argmax(y_all))
best_x = X_all[best_idx]
best_y = float(y_all[best_idx])

print(f"Best-so-far y={best_y:.6f} at idx={best_idx}")
print(f"Best-so-far x={np.round(best_x, 6)}")

# 3) Fit GP surrogate
gp = fit_gp(
    X_all,
    y_all,
    length_scale=0.2,
    kernel_amplitude=1.0,
    random_state=42,
)

# 4) Candidate pool (mostly global exploration)
n_candidates = int(cfg["n_candidates"])
local_frac = float(cfg["local_frac"])
local_sigma = float(cfg["local_sigma"])

n_local = int(round(n_candidates * local_frac))
n_global = n_candidates - n_local

rng = np.random.default_rng(42 + func_id)

X_global = rng.uniform(0.0, 1.0, size=(n_global, dim))

X_local = rng.normal(loc=best_x, scale=local_sigma, size=(n_local, dim))
X_local = np.clip(X_local, 0.0, np.nextafter(1.0, 0.0))

X_candidates = np.vstack([X_global, X_local])
X_candidates = np.minimum(X_candidates, np.nextafter(1.0, 0.0))

print(f"Candidates: global={n_global}, local={n_local}, total={len(X_candidates)}")

# 5) Acquisition (UCB)
mean, std = gp.predict(X_candidates, return_std=True)
scores = acquisition_ucb(mean, std, kappa=float(cfg["kappa"]))

# 6) Pick best candidate
x_next = X_candidates[int(np.argmax(scores))]

# 7) Format + validate for portal
submission = format_submission(x_next)
ok, msg = validate_submission_string(submission, dim=dim)
if not ok:
    raise ValueError(f"Invalid submission for Function {func_id}: {msg} -> {submission}")

print("\n✅ Function 1 Week 7 submission:")
print(submission)

Function 1 | points=16 | dim=2
Best-so-far y=0.000000 at idx=15
Best-so-far x=[0.724738 0.737753]
Candidates: global=25500, local=4500, total=30000

✅ Function 1 Week 7 submission:
0.882780-0.467661


## Funciton 2

In [9]:
# ==========================================================
# Function 1 — Week 7 suggestion (explore wide)
# ==========================================================
func_id = 1
cfg = WEEK7_STRATEGY[func_id]

# 1) Load dataset up to Week 6
X_all, y_all = load_xy_with_weeks_1_to_6(func_id, DATA_RAW_DIR)
dim = X_all.shape[1]

print(f"Function {func_id} | points={len(y_all)} | dim={dim}")

# 2) Best-so-far
best_idx = int(np.argmax(y_all))
best_x = X_all[best_idx]
best_y = float(y_all[best_idx])

print(f"Best-so-far y={best_y:.6f} at idx={best_idx}")
print(f"Best-so-far x={np.round(best_x, 6)}")

# 3) Fit GP surrogate
gp = fit_gp(
    X_all,
    y_all,
    length_scale=0.2,
    kernel_amplitude=1.0,
    random_state=42,
)

# 4) Candidate pool (mostly global exploration)
n_candidates = int(cfg["n_candidates"])
local_frac = float(cfg["local_frac"])
local_sigma = float(cfg["local_sigma"])

n_local = int(round(n_candidates * local_frac))
n_global = n_candidates - n_local

rng = np.random.default_rng(42 + func_id)

X_global = rng.uniform(0.0, 1.0, size=(n_global, dim))

X_local = rng.normal(loc=best_x, scale=local_sigma, size=(n_local, dim))
X_local = np.clip(X_local, 0.0, np.nextafter(1.0, 0.0))

X_candidates = np.vstack([X_global, X_local])
X_candidates = np.minimum(X_candidates, np.nextafter(1.0, 0.0))

print(f"Candidates: global={n_global}, local={n_local}, total={len(X_candidates)}")

# 5) Acquisition (UCB)
mean, std = gp.predict(X_candidates, return_std=True)
scores = acquisition_ucb(mean, std, kappa=float(cfg["kappa"]))

# 6) Pick best candidate
x_next = X_candidates[int(np.argmax(scores))]

# 7) Format + validate for portal
submission = format_submission(x_next)
ok, msg = validate_submission_string(submission, dim=dim)
if not ok:
    raise ValueError(f"Invalid submission for Function {func_id}: {msg} -> {submission}")

print("\n✅ Function 1 Week 7 submission:")
print(submission)

Function 1 | points=16 | dim=2
Best-so-far y=0.000000 at idx=15
Best-so-far x=[0.724738 0.737753]
Candidates: global=25500, local=4500, total=30000

✅ Function 1 Week 7 submission:
0.882780-0.467661


## Function 3

In [10]:
# ==========================================================
# Function 3 — Week 7 suggestion (balanced; avoid pure corner)
# ==========================================================
func_id = 3
cfg = WEEK7_STRATEGY[func_id]

# 1) Load dataset up to Week 6
X_all, y_all = load_xy_with_weeks_1_to_6(func_id, DATA_RAW_DIR)
dim = X_all.shape[1]

print(f"Function {func_id} | points={len(y_all)} | dim={dim}")

# 2) Best-so-far
best_idx = int(np.argmax(y_all))
best_x = X_all[best_idx]
best_y = float(y_all[best_idx])

print(f"Best-so-far y={best_y:.6f} at idx={best_idx}")
print(f"Best-so-far x={np.round(best_x, 6)}")

# 3) Fit GP surrogate
gp = fit_gp(
    X_all,
    y_all,
    length_scale=0.2,
    kernel_amplitude=1.0,
    random_state=42,
)

# 4) Candidate pool (balanced mix)
n_candidates = int(cfg["n_candidates"])
local_frac = float(cfg["local_frac"])
local_sigma = float(cfg["local_sigma"])

n_local = int(round(n_candidates * local_frac))
n_global = n_candidates - n_local

rng = np.random.default_rng(42 + func_id)

X_global = rng.uniform(0.0, 1.0, size=(n_global, dim))

X_local = rng.normal(loc=best_x, scale=local_sigma, size=(n_local, dim))
X_local = np.clip(X_local, 0.0, np.nextafter(1.0, 0.0))

X_candidates = np.vstack([X_global, X_local])
X_candidates = np.minimum(X_candidates, np.nextafter(1.0, 0.0))

print(f"Candidates: global={n_global}, local={n_local}, total={len(X_candidates)}")

# 5) Acquisition (UCB)
mean, std = gp.predict(X_candidates, return_std=True)
scores = acquisition_ucb(mean, std, kappa=float(cfg["kappa"]))

# Optional: small penalty for candidates too close to all-ones corner
near_corner = np.all(X_candidates >= 0.95, axis=1)
scores = scores - 0.5 * near_corner.astype(float)

# 6) Pick best candidate
x_next = X_candidates[int(np.argmax(scores))]

# 7) Format + validate for portal
submission = format_submission(x_next)
ok, msg = validate_submission_string(submission, dim=dim)
if not ok:
    raise ValueError(f"Invalid submission for Function {func_id}: {msg} -> {submission}")

print("\n✅ Function 3 Week 7 submission:")
print(submission)

Function 3 | points=21 | dim=3
Best-so-far y=-0.013221 at idx=15
Best-so-far x=[0.418575 0.366581 0.521443]
Candidates: global=13200, local=8800, total=22000

✅ Function 3 Week 7 submission:
0.384566-0.694037-0.001081


## Function 4

In [12]:
# ==========================================================
# Function 4 — Week 7 suggestion (local exploit around recovery region)
# ==========================================================
func_id = 4
cfg = WEEK7_STRATEGY[func_id]

# 1) Load dataset up to Week 6
X_all, y_all = load_xy_with_weeks_1_to_6(func_id, DATA_RAW_DIR)
dim = X_all.shape[1]

print(f"Function {func_id} | points={len(y_all)} | dim={dim}")

# 2) Best-so-far
best_idx = int(np.argmax(y_all))
best_x = X_all[best_idx]
best_y = float(y_all[best_idx])

print(f"Best-so-far y={best_y:.6f} at idx={best_idx}")
print(f"Best-so-far x={np.round(best_x, 6)}")

# 3) Fit GP surrogate
gp = fit_gp(
    X_all,
    y_all,
    length_scale=0.2,
    kernel_amplitude=1.0,
    random_state=42,
)

# 4) Candidate pool (more local than global)
n_candidates = int(cfg["n_candidates"])
local_frac = float(cfg["local_frac"])
local_sigma = float(cfg["local_sigma"])

n_local = int(round(n_candidates * local_frac))
n_global = n_candidates - n_local

rng = np.random.default_rng(42 + func_id)

X_global = rng.uniform(0.0, 1.0, size=(n_global, dim))

X_local = rng.normal(loc=best_x, scale=local_sigma, size=(n_local, dim))
X_local = np.clip(X_local, 0.0, np.nextafter(1.0, 0.0))

X_candidates = np.vstack([X_global, X_local])
X_candidates = np.minimum(X_candidates, np.nextafter(1.0, 0.0))

print(f"Candidates: global={n_global}, local={n_local}, total={len(X_candidates)}")

# 5) Acquisition (UCB)
mean, std = gp.predict(X_candidates, return_std=True)
scores = acquisition_ucb(mean, std, kappa=float(cfg["kappa"]))

# 6) Pick best candidate
x_next = X_candidates[int(np.argmax(scores))]

# 7) Format + validate for portal
submission = format_submission(x_next)
ok, msg = validate_submission_string(submission, dim=dim)
if not ok:
    raise ValueError(f"Invalid submission for Function {func_id}: {msg} -> {submission}")

print("\n✅ Function 4 Week 7 submission:")
print(submission)

Function 4 | points=36 | dim=4
Best-so-far y=-0.471335 at idx=30
Best-so-far x=[0.370752 0.426131 0.337047 0.465539]
Candidates: global=5400, local=12600, total=18000

✅ Function 4 Week 7 submission:
0.340322-0.429093-0.400312-0.470949


## Function 5

In [13]:
# ==========================================================
# Function 5 — Week 7 (SPECIAL STRATEGY: boundary control)
# ==========================================================
func_id = 5

# 1) Load dataset up to Week 6
X_all, y_all = load_xy_with_weeks_1_to_6(func_id, DATA_RAW_DIR)
dim = X_all.shape[1]

print(f"Function {func_id} | points={len(y_all)} | dim={dim}")

# 2) Best-so-far (for reference only)
best_idx = int(np.argmax(y_all))
best_x = X_all[best_idx]
best_y = float(y_all[best_idx])

print(f"Best-so-far y={best_y:.6f} at idx={best_idx}")
print(f"Best-so-far x={np.round(best_x, 6)}")

# ==========================================================
# 🔥 CORE IDEA:
# We DO NOT trust GP here.
# Instead, we probe slightly INSIDE boundary (not 0.999999)
# ==========================================================

# Strategy:
# - Avoid exact 1.0 (explodes)
# - Try slightly inside boundary
# - Explore multiple "rings" near 1

candidates = np.array([
    [0.97, 0.97, 0.97, 0.97],
    [0.96, 0.96, 0.96, 0.96],
    [0.95, 0.95, 0.95, 0.95],
    [0.98, 0.97, 0.96, 0.95],   # asymmetric probe
    [0.97, 0.98, 0.95, 0.96],
])

# Clip just in case
candidates = np.clip(candidates, 0.0, np.nextafter(1.0, 0.0))

# ==========================================================
# Optional: Evaluate using GP ONLY for ranking (not trusting absolute)
# ==========================================================
gp = fit_gp(
    X_all,
    y_all,
    length_scale=0.2,
    kernel_amplitude=1.0,
    random_state=42,
)

mean, std = gp.predict(candidates, return_std=True)

# Use LOW kappa → exploit GP ranking only
scores = mean + 0.5 * std

for i, (x, m, s) in enumerate(zip(candidates, mean, std)):
    print(f"Candidate {i}: x={np.round(x,6)} | mean={m:.3f} | std={s:.3f}")

# Pick best candidate
x_next = candidates[int(np.argmax(scores))]

# ==========================================================
# Format + validate
# ==========================================================
submission = format_submission(x_next)
ok, msg = validate_submission_string(submission, dim=dim)
if not ok:
    raise ValueError(f"Invalid submission for Function {func_id}: {msg} -> {submission}")

print("\n🚨 Function 5 Week 7 submission (controlled boundary):")
print(submission)

Function 5 | points=26 | dim=4
Best-so-far y=8662.405001 at idx=25
Best-so-far x=[0.999999 0.999999 0.999999 0.999999]
Candidate 0: x=[0.97 0.97 0.97 0.97] | mean=6520.348 | std=9.495
Candidate 1: x=[0.96 0.96 0.96 0.96] | mean=5832.442 | std=23.124
Candidate 2: x=[0.95 0.95 0.95 0.95] | mean=5162.546 | std=41.705
Candidate 3: x=[0.98 0.97 0.96 0.95] | mean=6013.646 | std=138.536
Candidate 4: x=[0.97 0.98 0.95 0.96] | mean=6220.708 | std=106.851

🚨 Function 5 Week 7 submission (controlled boundary):
0.970000-0.970000-0.970000-0.970000


## Function 6

In [14]:
# ==========================================================
# Function 6 — Week 7 suggestion (controlled local refinement)
# ==========================================================
func_id = 6
cfg = WEEK7_STRATEGY[func_id]

# 1) Load dataset up to Week 6
X_all, y_all = load_xy_with_weeks_1_to_6(func_id, DATA_RAW_DIR)
dim = X_all.shape[1]

print(f"Function {func_id} | points={len(y_all)} | dim={dim}")

# 2) Best-so-far
best_idx = int(np.argmax(y_all))
best_x = X_all[best_idx]
best_y = float(y_all[best_idx])

print(f"Best-so-far y={best_y:.6f} at idx={best_idx}")
print(f"Best-so-far x={np.round(best_x, 6)}")

# 3) Fit GP
gp = fit_gp(
    X_all,
    y_all,
    length_scale=0.15,   # tighter → more local sensitivity
    kernel_amplitude=1.0,
    random_state=42,
)

# ==========================================================
# 🔥 Key idea:
# - Strong local refinement
# - Keep last dimension near 0 (observed behaviour)
# ==========================================================

n_candidates = int(cfg["n_candidates"])
local_frac = float(cfg["local_frac"])
local_sigma = float(cfg["local_sigma"])

n_local = int(round(n_candidates * local_frac))
n_global = n_candidates - n_local

rng = np.random.default_rng(42 + func_id)

# Global exploration
X_global = rng.uniform(0.0, 1.0, size=(n_global, dim))

# Local exploration around best
X_local = rng.normal(loc=best_x, scale=local_sigma, size=(n_local, dim))

# 🔥 Important constraint: keep last dimension close to 0
X_local[:, -1] = np.clip(rng.normal(loc=0.0, scale=0.02, size=n_local), 0.0, 0.05)

# Clip bounds
X_local = np.clip(X_local, 0.0, np.nextafter(1.0, 0.0))

X_candidates = np.vstack([X_global, X_local])
X_candidates = np.minimum(X_candidates, np.nextafter(1.0, 0.0))

print(f"Candidates: global={n_global}, local={n_local}, total={len(X_candidates)}")

# 4) Acquisition (moderate exploration)
mean, std = gp.predict(X_candidates, return_std=True)
scores = acquisition_ucb(mean, std, kappa=float(cfg["kappa"]))

# 5) Pick best
x_next = X_candidates[int(np.argmax(scores))]

# 6) Format + validate
submission = format_submission(x_next)
ok, msg = validate_submission_string(submission, dim=dim)
if not ok:
    raise ValueError(f"Invalid submission for Function {func_id}: {msg} -> {submission}")

print("\n✅ Function 6 Week 7 submission:")
print(submission)

Function 6 | points=26 | dim=5
Best-so-far y=-0.498868 at idx=20
Best-so-far x=[0.296485 0.322087 0.332446 0.760368 0.040944]
Candidates: global=15600, local=8400, total=24000

✅ Function 6 Week 7 submission:
0.548860-0.343604-0.964802-0.922782-0.013847


## Function 7

In [15]:
# ==========================================================
# Function 7 — Week 7 suggestion (return to best region)
# ==========================================================
func_id = 7
cfg = WEEK7_STRATEGY[func_id]

# 1) Load dataset up to Week 6
X_all, y_all = load_xy_with_weeks_1_to_6(func_id, DATA_RAW_DIR)
dim = X_all.shape[1]

print(f"Function {func_id} | points={len(y_all)} | dim={dim}")

# 2) Best-so-far
best_idx = int(np.argmax(y_all))
best_x = X_all[best_idx]
best_y = float(y_all[best_idx])

print(f"Best-so-far y={best_y:.6f} at idx={best_idx}")
print(f"Best-so-far x={np.round(best_x, 6)}")

# 3) Fit GP surrogate
gp = fit_gp(
    X_all,
    y_all,
    length_scale=0.15,   # a bit tighter for local refinement
    kernel_amplitude=1.0,
    random_state=42,
)

# 4) Candidate pool (mostly local around best_x)
n_candidates = int(cfg["n_candidates"])
local_frac = float(cfg["local_frac"])
local_sigma = float(cfg["local_sigma"])

n_local = int(round(n_candidates * local_frac))
n_global = n_candidates - n_local

rng = np.random.default_rng(42 + func_id)

# Global candidates
X_global = rng.uniform(0.0, 1.0, size=(n_global, dim))

# Local candidates around best_x
X_local = rng.normal(loc=best_x, scale=local_sigma, size=(n_local, dim))
X_local = np.clip(X_local, 0.0, np.nextafter(1.0, 0.0))

# Combine + portal constraint
X_candidates = np.vstack([X_global, X_local])
X_candidates = np.minimum(X_candidates, np.nextafter(1.0, 0.0))

print(f"Candidates: global={n_global}, local={n_local}, total={len(X_candidates)}")

# 5) Acquisition (lower kappa = more exploit)
mean, std = gp.predict(X_candidates, return_std=True)
scores = acquisition_ucb(mean, std, kappa=float(cfg["kappa"]))

# Optional: slight penalty for candidates too far from best_x
# Helps avoid drifting away again
dist_from_best = np.sqrt(((X_candidates - best_x) ** 2).sum(axis=1))
scores = scores - 0.05 * dist_from_best

# 6) Pick best candidate
x_next = X_candidates[int(np.argmax(scores))]

# 7) Format + validate for portal
submission = format_submission(x_next)
ok, msg = validate_submission_string(submission, dim=dim)
if not ok:
    raise ValueError(f"Invalid submission for Function {func_id}: {msg} -> {submission}")

print("\n✅ Function 7 Week 7 submission:")
print(submission)

Function 7 | points=36 | dim=6
Best-so-far y=2.715108 at idx=34
Best-so-far x=[0.099617 0.221403 0.31364  0.206654 0.27198  0.685887]
Candidates: global=5400, local=12600, total=18000

✅ Function 7 Week 7 submission:
0.080227-0.198226-0.333285-0.278112-0.276086-0.656664


## Function 8

In [16]:
# ==========================================================
# Function 8 — Week 7 suggestion (balanced high-dim search)
# ==========================================================
func_id = 8
cfg = WEEK7_STRATEGY[func_id]

# 1) Load dataset up to Week 6
X_all, y_all = load_xy_with_weeks_1_to_6(func_id, DATA_RAW_DIR)
dim = X_all.shape[1]

print(f"Function {func_id} | points={len(y_all)} | dim={dim}")

# 2) Best-so-far
best_idx = int(np.argmax(y_all))
best_x = X_all[best_idx]
best_y = float(y_all[best_idx])

print(f"Best-so-far y={best_y:.6f} at idx={best_idx}")
print(f"Best-so-far x={np.round(best_x, 6)}")

# 3) Fit GP (slightly smoother for high-dim)
gp = fit_gp(
    X_all,
    y_all,
    length_scale=0.25,   # smoother → avoids overfitting noise
    kernel_amplitude=1.0,
    random_state=42,
)

# ==========================================================
# 🔥 Strategy:
# - High-dim → need diversity
# - Balanced global + local
# - Avoid collapsing to one region
# ==========================================================

n_candidates = int(cfg["n_candidates"])
local_frac = float(cfg["local_frac"])
local_sigma = float(cfg["local_sigma"])

n_local = int(round(n_candidates * local_frac))
n_global = n_candidates - n_local

rng = np.random.default_rng(42 + func_id)

# Global exploration (important in 8D)
X_global = rng.uniform(0.0, 1.0, size=(n_global, dim))

# Local refinement
X_local = rng.normal(loc=best_x, scale=local_sigma, size=(n_local, dim))
X_local = np.clip(X_local, 0.0, np.nextafter(1.0, 0.0))

# Combine
X_candidates = np.vstack([X_global, X_local])
X_candidates = np.minimum(X_candidates, np.nextafter(1.0, 0.0))

print(f"Candidates: global={n_global}, local={n_local}, total={len(X_candidates)}")

# 4) Acquisition (balanced exploration)
mean, std = gp.predict(X_candidates, return_std=True)
scores = acquisition_ucb(mean, std, kappa=float(cfg["kappa"]))

# Optional: diversity bonus (encourage spread)
dist_from_best = np.sqrt(((X_candidates - best_x) ** 2).sum(axis=1))
scores = scores + 0.02 * dist_from_best

# 5) Pick best
x_next = X_candidates[int(np.argmax(scores))]

# 6) Format + validate
submission = format_submission(x_next)
ok, msg = validate_submission_string(submission, dim=dim)
if not ok:
    raise ValueError(f"Invalid submission for Function {func_id}: {msg} -> {submission}")

print("\n✅ Function 8 Week 7 submission:")
print(submission)

Function 8 | points=46 | dim=8
Best-so-far y=9.894432 at idx=40
Best-so-far x=[0.147947 0.193825 0.060981 0.008004 0.719306 0.453461 0.045592 0.93897 ]
Candidates: global=11000, local=11000, total=22000

✅ Function 8 Week 7 submission:
0.113137-0.127176-0.089091-0.095525-0.146696-0.007133-0.173747-0.999806
